# Body-centered keypoint extraction

This version extracts all 33 MediaPipe Pose landmarks, normalizes x/y/z around the body, and saves each dataset split as CSV.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from pathlib import Path
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
# Resolve paths whether the notebook starts in the project root or code/.
project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

model_path = project_root / "model" / "mediapipe" / "pose_landmarker_lite.task"
if not model_path.is_file():
    raise FileNotFoundError(f"MediaPipe model not found: {model_path}")

options = vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=str(model_path)),
    running_mode=vision.RunningMode.IMAGE,
    num_poses=1,
    output_segmentation_masks=False,
)
pose_detector = vision.PoseLandmarker.create_from_options(options)

In [ ]:
# Base directory setup
base_dir = project_root / "data" / "test"
output_dir = project_root / "data" / "processed"
if not base_dir.is_dir():
    raise FileNotFoundError(f"Input directory not found: {base_dir}")
output_dir.mkdir(parents=True, exist_ok=True)

splits = ["TRAIN", "TEST"]
categories = ["downdog", "goddess", "plank", "tree", "warrior2"]

In [ ]:
# MediaPipe indices: shoulders 11/12, hips 23/24.
# x, y and z are centered and scaled; visibility is used only for filtering.
CONFIDENCE_THRESHOLD = 0.25
TORSO_MULTIPLIER = 2.5

def normalize_keypoints(keypoints, visibility, confidence_threshold=CONFIDENCE_THRESHOLD):
    """Normalize a (33, 3) [x, y, z] MediaPipe pose.

    The hip midpoint is the preferred origin. If the hips are unreliable,
    the shoulder midpoint or the mean of reliable joints is used. The scale
    is the larger of 2.5 torso lengths and the furthest reliable joint.
    Low-visibility coordinates are set to zero.
    """
    normalized = keypoints.astype(np.float32, copy=True)
    xy = normalized[:, :2]
    z = normalized[:, 2]
    reliable = visibility >= confidence_threshold

    if not reliable.any():
        normalized[:, :] = 0.0
        return normalized, "none"

    hips_reliable = reliable[23] and reliable[24]
    shoulders_reliable = reliable[11] and reliable[12]

    if hips_reliable:
        center = (xy[23] + xy[24]) / 2.0
        center_z = (z[23] + z[24]) / 2.0
        center_type = "hip_center"
    elif shoulders_reliable:
        center = (xy[11] + xy[12]) / 2.0
        center_z = (z[11] + z[12]) / 2.0
        center_type = "shoulder_center"
    else:
        center = xy[reliable].mean(axis=0)
        center_z = z[reliable].mean()
        center_type = "full_body_center"

    body_radius = np.linalg.norm(xy[reliable] - center, axis=1).max()

    if hips_reliable and shoulders_reliable:
        shoulder_center = (xy[11] + xy[12]) / 2.0
        hip_center = (xy[23] + xy[24]) / 2.0
        torso_size = np.linalg.norm(shoulder_center - hip_center)
        pose_scale = max(body_radius, TORSO_MULTIPLIER * torso_size)
    else:
        pose_scale = body_radius

    # Degenerate detections should not create infinities or huge values.
    if pose_scale <= np.finfo(np.float32).eps:
        normalized[:, :] = 0.0
        return normalized, "none"

    normalized[:, :2] = (xy - center) / pose_scale
    normalized[:, 2] = (z - center_z) / pose_scale
    normalized[~reliable, :] = 0.0
    return normalized, center_type

In [ ]:
# Keep all 33 MediaPipe Pose landmarks in their native index order.
LANDMARK_COUNT = 33
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
kp_cols = [f"kp_{i}_{axis}" for i in range(LANDMARK_COUNT) for axis in ("x", "y", "z")]
rows_by_split = {}

for split in splits:
    rows = []

    for category in categories:
        folder_path = base_dir / split / category
        if not folder_path.exists():
            continue

        for img_path in sorted(folder_path.iterdir()):
            if not img_path.is_file() or img_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            image_bgr = cv2.imread(str(img_path))
            if image_bgr is None:
                print(f"Warning: unable to read {img_path}")
                continue

            height, width = image_bgr.shape[:2]
            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            result = pose_detector.detect(mp_image)

            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]
                keypoints = np.asarray(
                    [
                        (
                            landmarks[index].x * width,
                            landmarks[index].y * height,
                            landmarks[index].z * width,
                        )
                        for index in range(LANDMARK_COUNT)
                    ],
                    dtype=np.float32,
                )
                visibility = np.asarray(
                    [landmarks[index].visibility for index in range(LANDMARK_COUNT)],
                    dtype=np.float32,
                )
                normalized_keypoints, center_type = normalize_keypoints(keypoints, visibility)
                kp_data = normalized_keypoints.reshape(-1)
            else:
                kp_data = np.full(LANDMARK_COUNT * 3, np.nan, dtype=np.float32)
                center_type = "none"

            image_name = (Path("data") / split / category / img_path.name).as_posix()
            rows.append([image_name, width, height, *kp_data, center_type, category])

    rows_by_split[split] = rows

In [ ]:
# Create and save one DataFrame per dataset split.
columns = ["image_name", "width", "height"] + kp_cols + ["center_type", "label"]

for split, rows in rows_by_split.items():
    df = pd.DataFrame(rows, columns=columns)
    output_path = output_dir / f"keypoints_{split.lower()}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved {split} set keypoints ({len(df)} images) to: {output_path}")

pose_detector.close()